## Deploying a CPLEX LP file with watsonx.ai Runtime

This notebook shows you how to deploy a CPLEX LP file, create and monitor jobs using the watsonx.ai Python Client.

This notebook runs on Python.

**Table of contents:**

- [Set up the watsonx.ai client](#setup)
- [Create a client instance](#create)
- [Upload your model on watsonx.ai Runtime](#upload)
- [Create a deployment](#deploy)
- [Create and monitor a job with inline data for your deployed model](#job)

<a id='setup'></a>
### Set up the watsonx.ai client

Before you use the sample code in this notebook, you must:

- create a <a href="https://cloud.ibm.com/catalog?category=ai" target="_blank" rel="noopener noreferrer">watsonx.ai Runtime Service</a> instance. A free plan is offered and information about how to create the instance can be found at <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas" target="_blank" rel="noopener noreferrer"> https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas.</a>


Install and import the watsonx.ai client library.

In [ ]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

<a id='create'></a>
### Create a client instance

Use your IBM Cloud API key. You can find information on how to get your API key <a href="https://dataplatform.cloud.ibm.com/docs/content/DO/WML_Deployment/DeployModelRest.html?audience=wdp&context=cpdaas#tasktask_deploymodelREST__prereq_el2_nft_bhb">here</a> and the instance URL <a href="https://cloud.ibm.com/apidocs/machine-learning#endpoint-url">here</a>.

In [ ]:
# Instantiate a client using credentials
credentials = Credentials(
      api_key = "<API_key>",
      url = "<instance_url>"
)

client = APIClient(credentials)

In [ ]:
client.version

<a id='upload'></a>
### Upload your model on watsonx.ai Runtime

Store model in watsonx.ai Runtime with metadata including the model type and runtime.

Get the `model_uid`.

In [ ]:
# Find the space ID

space_name = "<space_name>"

space_id = [x['metadata']['id'] for x in client.spaces.get_details()['resources'] if x['entity']['name'] == space_name][0]

client = APIClient(credentials, space_id = space_id)

In [ ]:
mnist_metadata = {
    client.repository.ModelMetaNames.NAME: "LP",
    client.repository.ModelMetaNames.DESCRIPTION: "Model for LP",
    client.repository.ModelMetaNames.TYPE: "do-cplex_22.1",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: client.software_specifications.get_id_by_name("do_22.1"),
}

model_details = client.repository.store_model(meta_props=mnist_metadata)
model_uid = client.repository.get_model_id(model_details)

<a id='deploy'></a>
### Create a deployment 

Create a batch deployment for the model, providing information such as:
* the maximum number of compute nodes
* the T-shirt size of the compute nodes

Get the `deployment_uid`.

In [ ]:
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "LP Deployment",
    client.deployments.ConfigurationMetaNames.DESCRIPTION: "LP Deployment",
    client.deployments.ConfigurationMetaNames.BATCH: {},
    client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {'name': 'S', 'num_nodes': 1}
}

deployment_details = client.deployments.create(model_uid, meta_props=meta_props)

deployment_uid = client.deployments.get_id(deployment_details)

# print deployment id if needed
# print( deployment_uid )

In [ ]:
# List all existing deployments

client.deployments.list()

<a id='job'></a>
### Create and monitor a job with inline data for your deployed model

Create a payload containing inline input data.

Create a new job with this payload and the deployment.

Get the `job_uid`.

In [ ]:
import base64

sample_string='''
\\ENCODING=ISO-8859-1
\\Problem name: IloCplex

Maximize
 obj: assign_Mark_deal1 + assign_Mark_deal2 + assign_Mark_deal3
      + assign_Mark_deal4 + assign_Mark_deal5 + assign_Mark_deal6
      + assign_Mark_deal7 + assign_Mark_deal8 + assign_Mark_deal9
      + assign_Mark_deal10 + assign_Steve_deal1 + assign_Steve_deal2
      + assign_Steve_deal3 + assign_Steve_deal4 + assign_Steve_deal5
      + assign_Steve_deal6 + assign_Steve_deal7 + assign_Steve_deal8
      + assign_Steve_deal9 + assign_Steve_deal10 + assign_Paul_deal1
      + assign_Paul_deal2 + assign_Paul_deal3 + assign_Paul_deal4
      + assign_Paul_deal5 + assign_Paul_deal6 + assign_Paul_deal7
      + assign_Paul_deal8 + assign_Paul_deal9 + assign_Paul_deal10
      + assign_John_deal1 + assign_John_deal2 + assign_John_deal3
      + assign_John_deal4 + assign_John_deal5 + assign_John_deal6
      + assign_John_deal7 + assign_John_deal8 + assign_John_deal9
      + assign_John_deal10 + assign_Jack_deal1 + assign_Jack_deal2
      + assign_Jack_deal3 + assign_Jack_deal4 + assign_Jack_deal5
      + assign_Jack_deal6 + assign_Jack_deal7 + assign_Jack_deal8
      + assign_Jack_deal9 + assign_Jack_deal10 + assign_Eric_deal1
      + assign_Eric_deal2 + assign_Eric_deal3 + assign_Eric_deal4
      + assign_Eric_deal5 + assign_Eric_deal6 + assign_Eric_deal7
      + assign_Eric_deal8 + assign_Eric_deal9 + assign_Eric_deal10 + x108
Subject To
 c1:  x1 + x12 + x24 + x36 + x48 + x60 <= 20
 c2:  1000 assign_Mark_deal1 + 500 assign_Mark_deal2 + 800 assign_Mark_deal3
      + 500 assign_Mark_deal4 + 500 assign_Mark_deal5 + 300 assign_Mark_deal6
      + 500 assign_Mark_deal7 + 400 assign_Mark_deal8 + 500 assign_Mark_deal9
      + 500 assign_Mark_deal10 + 1000 assign_Steve_deal1
      + 500 assign_Steve_deal2 + 800 assign_Steve_deal3
      + 500 assign_Steve_deal4 + 500 assign_Steve_deal5
      + 300 assign_Steve_deal6 + 500 assign_Steve_deal7
      + 400 assign_Steve_deal8 + 500 assign_Steve_deal9
      + 500 assign_Steve_deal10 + 1000 assign_Paul_deal1
      + 500 assign_Paul_deal2 + 800 assign_Paul_deal3 + 500 assign_Paul_deal4
      + 500 assign_Paul_deal5 + 300 assign_Paul_deal6 + 500 assign_Paul_deal7
      + 400 assign_Paul_deal8 + 500 assign_Paul_deal9 + 500 assign_Paul_deal10
      + 1000 assign_John_deal1 + 500 assign_John_deal2 + 800 assign_John_deal3
      + 500 assign_John_deal4 + 500 assign_John_deal5 + 300 assign_John_deal6
      + 500 assign_John_deal7 + 400 assign_John_deal8 + 500 assign_John_deal9
      + 500 assign_John_deal10 + 1000 assign_Jack_deal1 + 500 assign_Jack_deal2
      + 800 assign_Jack_deal3 + 500 assign_Jack_deal4 + 500 assign_Jack_deal5
      + 300 assign_Jack_deal6 + 500 assign_Jack_deal7 + 400 assign_Jack_deal8
      + 500 assign_Jack_deal9 + 500 assign_Jack_deal10 + 1000 assign_Eric_deal1
      + 500 assign_Eric_deal2 + 800 assign_Eric_deal3 + 500 assign_Eric_deal4
      + 500 assign_Eric_deal5 + 300 assign_Eric_deal6 + 500 assign_Eric_deal7
      + 400 assign_Eric_deal8 + 500 assign_Eric_deal9 + 500 assign_Eric_deal10
      <= 5000
 c3:  assign_John_deal1 + assign_John_deal2 + assign_John_deal3
      + assign_John_deal4 + assign_John_deal5 + assign_John_deal6
      + assign_John_deal7 + assign_John_deal8 + assign_John_deal9
      + assign_John_deal10 <= 3
 c4:  assign_Jack_deal1 + assign_Jack_deal2 + assign_Jack_deal3
      + assign_Jack_deal4 + assign_Jack_deal5 + assign_Jack_deal6
      + assign_Jack_deal7 + assign_Jack_deal8 + assign_Jack_deal9
      + assign_Jack_deal10 <= 3
 c5:  assign_Paul_deal1 + assign_Paul_deal2 + assign_Paul_deal3
      + assign_Paul_deal4 + assign_Paul_deal5 + assign_Paul_deal6
      + assign_Paul_deal7 + assign_Paul_deal8 + assign_Paul_deal9
      + assign_Paul_deal10 <= 3
 c6:  assign_Mark_deal1 + assign_Mark_deal2 + assign_Mark_deal3
      + assign_Mark_deal4 + assign_Mark_deal5 + assign_Mark_deal6
      + assign_Mark_deal7 + assign_Mark_deal8 + assign_Mark_deal9
      + assign_Mark_deal10 <= 3
 c7:  assign_Eric_deal1 + assign_Eric_deal2 + assign_Eric_deal3
      + assign_Eric_deal4 + assign_Eric_deal5 + assign_Eric_deal6
      + assign_Eric_deal7 + assign_Eric_deal8 + assign_Eric_deal9
      + assign_Eric_deal10 <= 3
 c8:  assign_Steve_deal1 + assign_Steve_deal2 + assign_Steve_deal3
      + assign_Steve_deal4 + assign_Steve_deal5 + assign_Steve_deal6
      + assign_Steve_deal7 + assign_Steve_deal8 + assign_Steve_deal9
      + assign_Steve_deal10 <= 3
 c9:  100000 assign_John_deal1 + 200000 assign_John_deal2
      + 150000 assign_John_deal3 + 150000 assign_John_deal4
      + 250000 assign_John_deal5 + 80000 assign_John_deal6
      + 50000 assign_John_deal7 + 100000 assign_John_deal8
      + 250000 assign_John_deal9 + 150000 assign_John_deal10 <= 250000
 c10: 100000 assign_Jack_deal1 + 200000 assign_Jack_deal2
      + 150000 assign_Jack_deal3 + 150000 assign_Jack_deal4
      + 250000 assign_Jack_deal5 + 80000 assign_Jack_deal6
      + 50000 assign_Jack_deal7 + 100000 assign_Jack_deal8
      + 250000 assign_Jack_deal9 + 150000 assign_Jack_deal10 <= 250000
 c11: 100000 assign_Paul_deal1 + 200000 assign_Paul_deal2
      + 150000 assign_Paul_deal3 + 150000 assign_Paul_deal4
      + 250000 assign_Paul_deal5 + 80000 assign_Paul_deal6
      + 50000 assign_Paul_deal7 + 100000 assign_Paul_deal8
      + 250000 assign_Paul_deal9 + 150000 assign_Paul_deal10 <= 250000
 c12: 100000 assign_Mark_deal1 + 200000 assign_Mark_deal2
      + 150000 assign_Mark_deal3 + 150000 assign_Mark_deal4
      + 250000 assign_Mark_deal5 + 80000 assign_Mark_deal6
      + 50000 assign_Mark_deal7 + 100000 assign_Mark_deal8
      + 250000 assign_Mark_deal9 + 150000 assign_Mark_deal10 <= 250000
 c13: 100000 assign_Eric_deal1 + 200000 assign_Eric_deal2
      + 150000 assign_Eric_deal3 + 150000 assign_Eric_deal4
      + 250000 assign_Eric_deal5 + 80000 assign_Eric_deal6
      + 50000 assign_Eric_deal7 + 100000 assign_Eric_deal8
      + 250000 assign_Eric_deal9 + 150000 assign_Eric_deal10 <= 250000
 c14: 100000 assign_Steve_deal1 + 200000 assign_Steve_deal2
      + 150000 assign_Steve_deal3 + 150000 assign_Steve_deal4
      + 250000 assign_Steve_deal5 + 80000 assign_Steve_deal6
      + 50000 assign_Steve_deal7 + 100000 assign_Steve_deal8
      + 250000 assign_Steve_deal9 + 150000 assign_Steve_deal10 <= 250000
 c15: assign_Mark_deal1 + assign_Steve_deal1 + assign_Paul_deal1
      + assign_John_deal1 + assign_Jack_deal1 + assign_Eric_deal1 <= 1
 c16: assign_Mark_deal2 + assign_Steve_deal2 + assign_Paul_deal2
      + assign_John_deal2 + assign_Jack_deal2 + assign_Eric_deal2 <= 1
 c17: assign_Mark_deal3 + assign_Steve_deal3 + assign_Paul_deal3
      + assign_John_deal3 + assign_Jack_deal3 + assign_Eric_deal3 <= 1
 c18: assign_Mark_deal4 + assign_Steve_deal4 + assign_Paul_deal4
      + assign_John_deal4 + assign_Jack_deal4 + assign_Eric_deal4 <= 1
 c19: assign_Mark_deal5 + assign_Steve_deal5 + assign_Paul_deal5
      + assign_John_deal5 + assign_Jack_deal5 + assign_Eric_deal5 <= 1
 c20: assign_Mark_deal6 + assign_Steve_deal6 + assign_Paul_deal6
      + assign_John_deal6 + assign_Jack_deal6 + assign_Eric_deal6 <= 1
 c21: assign_Mark_deal7 + assign_Steve_deal7 + assign_Paul_deal7
      + assign_John_deal7 + assign_Jack_deal7 + assign_Eric_deal7 <= 1
 c22: assign_Mark_deal8 + assign_Steve_deal8 + assign_Paul_deal8
      + assign_John_deal8 + assign_Jack_deal8 + assign_Eric_deal8 <= 1
 c23: assign_Mark_deal9 + assign_Steve_deal9 + assign_Paul_deal9
      + assign_John_deal9 + assign_Jack_deal9 + assign_Eric_deal9 <= 1
 c24: assign_Mark_deal10 + assign_Steve_deal10 + assign_Paul_deal10
      + assign_John_deal10 + assign_Jack_deal10 + assign_Eric_deal10 <= 1
 c25: assign_John_deal1 + assign_John_deal2 + assign_John_deal3
      + assign_John_deal4 + assign_John_deal5 + assign_John_deal6
      + assign_John_deal7 + assign_John_deal8 + assign_John_deal9
      + assign_John_deal10 <= 2
 c26: assign_Jack_deal1 + assign_Jack_deal2 + assign_Jack_deal3
      + assign_Jack_deal4 + assign_Jack_deal5 + assign_Jack_deal6
      + assign_Jack_deal7 + assign_Jack_deal8 + assign_Jack_deal9
      + assign_Jack_deal10 <= 2
 c27: assign_Paul_deal1 + assign_Paul_deal2 + assign_Paul_deal3
      + assign_Paul_deal4 + assign_Paul_deal5 + assign_Paul_deal6
      + assign_Paul_deal7 + assign_Paul_deal8 + assign_Paul_deal9
      + assign_Paul_deal10 <= 3
 c28: assign_Mark_deal1 + assign_Mark_deal2 + assign_Mark_deal3
      + assign_Mark_deal4 + assign_Mark_deal5 + assign_Mark_deal6
      + assign_Mark_deal7 + assign_Mark_deal8 + assign_Mark_deal9
      + assign_Mark_deal10 <= 2
 c29: assign_Eric_deal1 + assign_Eric_deal2 + assign_Eric_deal3
      + assign_Eric_deal4 + assign_Eric_deal5 + assign_Eric_deal6
      + assign_Eric_deal7 + assign_Eric_deal8 + assign_Eric_deal9
      + assign_Eric_deal10 <= 3
 c30: assign_Steve_deal1 + assign_Steve_deal2 + assign_Steve_deal3
      + assign_Steve_deal4 + assign_Steve_deal5 + assign_Steve_deal6
      + assign_Steve_deal7 + assign_Steve_deal8 + assign_Steve_deal9
      + assign_Steve_deal10 <= 2
 c31: x72 + x74 + x76 + x78 + x80 + x82 <= 3
 c32: x84 + x86 + x88 + x90 + x92 + x94 <= 3
 c33: x96 + x98 + x100 + x102 + x104 + x106 <= 3
 c34: 100000 assign_Mark_deal1 + 200000 assign_Mark_deal2
      + 150000 assign_Mark_deal3 + 150000 assign_Mark_deal4
      + 250000 assign_Mark_deal5 + 80000 assign_Mark_deal6
      + 50000 assign_Mark_deal7 + 100000 assign_Mark_deal8
      + 250000 assign_Mark_deal9 + 150000 assign_Mark_deal10
      + 100000 assign_Steve_deal1 + 200000 assign_Steve_deal2
      + 150000 assign_Steve_deal3 + 150000 assign_Steve_deal4
      + 250000 assign_Steve_deal5 + 80000 assign_Steve_deal6
      + 50000 assign_Steve_deal7 + 100000 assign_Steve_deal8
      + 250000 assign_Steve_deal9 + 150000 assign_Steve_deal10
      + 100000 assign_Paul_deal1 + 200000 assign_Paul_deal2
      + 150000 assign_Paul_deal3 + 150000 assign_Paul_deal4
      + 250000 assign_Paul_deal5 + 80000 assign_Paul_deal6
      + 50000 assign_Paul_deal7 + 100000 assign_Paul_deal8
      + 250000 assign_Paul_deal9 + 150000 assign_Paul_deal10
      + 100000 assign_John_deal1 + 200000 assign_John_deal2
      + 150000 assign_John_deal3 + 150000 assign_John_deal4
      + 250000 assign_John_deal5 + 80000 assign_John_deal6
      + 50000 assign_John_deal7 + 100000 assign_John_deal8
      + 250000 assign_John_deal9 + 150000 assign_John_deal10
      + 100000 assign_Jack_deal1 + 200000 assign_Jack_deal2
      + 150000 assign_Jack_deal3 + 150000 assign_Jack_deal4
      + 250000 assign_Jack_deal5 + 80000 assign_Jack_deal6
      + 50000 assign_Jack_deal7 + 100000 assign_Jack_deal8
      + 250000 assign_Jack_deal9 + 150000 assign_Jack_deal10
      + 100000 assign_Eric_deal1 + 200000 assign_Eric_deal2
      + 150000 assign_Eric_deal3 + 150000 assign_Eric_deal4
      + 250000 assign_Eric_deal5 + 80000 assign_Eric_deal6
      + 50000 assign_Eric_deal7 + 100000 assign_Eric_deal8
      + 250000 assign_Eric_deal9 + 150000 assign_Eric_deal10 <= 1000000
 c35: assign_Paul_deal1  = 0
 c36: assign_Mark_deal1  = 0
 c37: assign_Eric_deal1  = 0
 c38: assign_Steve_deal1  = 0
 c39: assign_Paul_deal2  = 0
 c40: assign_Mark_deal2  = 0
 c41: assign_Eric_deal2  = 0
 c42: assign_Steve_deal2  = 0
 c43: assign_Paul_deal3  = 0
 c44: assign_Mark_deal3  = 0
 c45: assign_Eric_deal3  = 0
 c46: assign_Steve_deal3  = 0
 c47: assign_John_deal4  = 0
 c48: assign_Jack_deal4  = 0
 c49: assign_Eric_deal4  = 0
 c50: assign_Steve_deal4  = 0
 c51: assign_John_deal5  = 0
 c52: assign_Jack_deal5  = 0
 c53: assign_Eric_deal5  = 0
 c54: assign_Steve_deal5  = 0
 c55: assign_John_deal6  = 0
 c56: assign_Jack_deal6  = 0
 c57: assign_Eric_deal6  = 0
 c58: assign_Steve_deal6  = 0
 c59: assign_John_deal7  = 0
 c60: assign_Jack_deal7  = 0
 c61: assign_Eric_deal7  = 0
 c62: assign_Steve_deal7  = 0
 c63: assign_John_deal8  = 0
 c64: assign_Jack_deal8  = 0
 c65: assign_Paul_deal8  = 0
 c66: assign_Mark_deal8  = 0
 c67: assign_John_deal9  = 0
 c68: assign_Jack_deal9  = 0
 c69: assign_Paul_deal9  = 0
 c70: assign_Mark_deal9  = 0
 c71: assign_John_deal10  = 0
 c72: assign_Jack_deal10  = 0
 c73: assign_Paul_deal10  = 0
 c74: assign_Mark_deal10  = 0
 i1:  x1 = 1 <-> assign_Mark_deal1 + assign_Mark_deal2 + assign_Mark_deal3
      + assign_Mark_deal4 + assign_Mark_deal5 + assign_Mark_deal6
      + assign_Mark_deal7 + assign_Mark_deal8 + assign_Mark_deal9
      + assign_Mark_deal10 >= 1
 i2:  x12 = 1 <-> assign_Steve_deal1 + assign_Steve_deal2 + assign_Steve_deal3
      + assign_Steve_deal4 + assign_Steve_deal5 + assign_Steve_deal6
      + assign_Steve_deal7 + assign_Steve_deal8 + assign_Steve_deal9
      + assign_Steve_deal10 >= 1
 i3:  x24 = 1 <-> assign_Paul_deal1 + assign_Paul_deal2 + assign_Paul_deal3
      + assign_Paul_deal4 + assign_Paul_deal5 + assign_Paul_deal6
      + assign_Paul_deal7 + assign_Paul_deal8 + assign_Paul_deal9
      + assign_Paul_deal10 >= 1
 i4:  x36 = 1 <-> assign_John_deal1 + assign_John_deal2 + assign_John_deal3
      + assign_John_deal4 + assign_John_deal5 + assign_John_deal6
      + assign_John_deal7 + assign_John_deal8 + assign_John_deal9
      + assign_John_deal10 >= 1
 i5:  x48 = 1 <-> assign_Jack_deal1 + assign_Jack_deal2 + assign_Jack_deal3
      + assign_Jack_deal4 + assign_Jack_deal5 + assign_Jack_deal6
      + assign_Jack_deal7 + assign_Jack_deal8 + assign_Jack_deal9
      + assign_Jack_deal10 >= 1
 i6:  x60 = 1 <-> assign_Eric_deal1 + assign_Eric_deal2 + assign_Eric_deal3
      + assign_Eric_deal4 + assign_Eric_deal5 + assign_Eric_deal6
      + assign_Eric_deal7 + assign_Eric_deal8 + assign_Eric_deal9
      + assign_Eric_deal10 >= 1
 i7:  x72 = 1 <-> assign_Mark_deal1 + assign_Mark_deal3 + assign_Mark_deal2
      >= 1
 i8:  x74 = 1 <-> assign_Paul_deal2 + assign_Paul_deal1 + assign_Paul_deal3
      >= 1
 i9:  x76 = 1 <-> assign_Steve_deal1 + assign_Steve_deal3 + assign_Steve_deal2
      >= 1
 i10: x78 = 1 <-> assign_John_deal3 + assign_John_deal2 + assign_John_deal1
      >= 1
 i11: x80 = 1 <-> assign_Jack_deal1 + assign_Jack_deal2 + assign_Jack_deal3
      >= 1
 i12: x82 = 1 <-> assign_Eric_deal1 + assign_Eric_deal3 + assign_Eric_deal2
      >= 1
 i13: x84 = 1 <-> assign_Mark_deal6 + assign_Mark_deal4 + assign_Mark_deal7
      + assign_Mark_deal5 >= 1
 i14: x86 = 1 <-> assign_Paul_deal5 + assign_Paul_deal4 + assign_Paul_deal7
      + assign_Paul_deal6 >= 1
 i15: x88 = 1 <-> assign_Steve_deal4 + assign_Steve_deal5 + assign_Steve_deal6
      + assign_Steve_deal7 >= 1
 i16: x90 = 1 <-> assign_John_deal6 + assign_John_deal4 + assign_John_deal5
      + assign_John_deal7 >= 1
 i17: x92 = 1 <-> assign_Jack_deal4 + assign_Jack_deal6 + assign_Jack_deal5
      + assign_Jack_deal7 >= 1
 i18: x94 = 1 <-> assign_Eric_deal4 + assign_Eric_deal5 + assign_Eric_deal6
      + assign_Eric_deal7 >= 1
 i19: x96 = 1 <-> assign_Mark_deal8 + assign_Mark_deal10 + assign_Mark_deal9
      >= 1
 i20: x98 = 1 <-> assign_Steve_deal10 + assign_Steve_deal8 + assign_Steve_deal9
      >= 1
 i21: x100 = 1 <-> assign_Paul_deal9 + assign_Paul_deal8 + assign_Paul_deal10
      >= 1
 i22: x102 = 1 <-> assign_John_deal8 + assign_John_deal9 + assign_John_deal10
      >= 1
 i23: x104 = 1 <-> assign_Jack_deal8 + assign_Jack_deal10 + assign_Jack_deal9
      >= 1
 i24: x106 = 1 <-> assign_Eric_deal9 + assign_Eric_deal8 + assign_Eric_deal10
      >= 1
Bounds
 0 <= x1 <= 1
 0 <= assign_Mark_deal1 <= 1
 0 <= assign_Mark_deal2 <= 1
 0 <= assign_Mark_deal3 <= 1
 0 <= assign_Mark_deal4 <= 1
 0 <= assign_Mark_deal5 <= 1
 0 <= assign_Mark_deal6 <= 1
 0 <= assign_Mark_deal7 <= 1
 0 <= assign_Mark_deal8 <= 1
 0 <= assign_Mark_deal9 <= 1
 0 <= assign_Mark_deal10 <= 1
 0 <= x12 <= 1
 0 <= assign_Steve_deal1 <= 1
 0 <= assign_Steve_deal2 <= 1
 0 <= assign_Steve_deal3 <= 1
 0 <= assign_Steve_deal4 <= 1
 0 <= assign_Steve_deal5 <= 1
 0 <= assign_Steve_deal6 <= 1
 0 <= assign_Steve_deal7 <= 1
 0 <= assign_Steve_deal8 <= 1
 0 <= assign_Steve_deal9 <= 1
 0 <= assign_Steve_deal10 <= 1
 0 <= x24 <= 1
 0 <= assign_Paul_deal1 <= 1
 0 <= assign_Paul_deal2 <= 1
 0 <= assign_Paul_deal3 <= 1
 0 <= assign_Paul_deal4 <= 1
 0 <= assign_Paul_deal5 <= 1
 0 <= assign_Paul_deal6 <= 1
 0 <= assign_Paul_deal7 <= 1
 0 <= assign_Paul_deal8 <= 1
 0 <= assign_Paul_deal9 <= 1
 0 <= assign_Paul_deal10 <= 1
 0 <= x36 <= 1
 0 <= assign_John_deal1 <= 1
 0 <= assign_John_deal2 <= 1
 0 <= assign_John_deal3 <= 1
 0 <= assign_John_deal4 <= 1
 0 <= assign_John_deal5 <= 1
 0 <= assign_John_deal6 <= 1
 0 <= assign_John_deal7 <= 1
 0 <= assign_John_deal8 <= 1
 0 <= assign_John_deal9 <= 1
 0 <= assign_John_deal10 <= 1
 0 <= x48 <= 1
 0 <= assign_Jack_deal1 <= 1
 0 <= assign_Jack_deal2 <= 1
 0 <= assign_Jack_deal3 <= 1
 0 <= assign_Jack_deal4 <= 1
 0 <= assign_Jack_deal5 <= 1
 0 <= assign_Jack_deal6 <= 1
 0 <= assign_Jack_deal7 <= 1
 0 <= assign_Jack_deal8 <= 1
 0 <= assign_Jack_deal9 <= 1
 0 <= assign_Jack_deal10 <= 1
 0 <= x60 <= 1
 0 <= assign_Eric_deal1 <= 1
 0 <= assign_Eric_deal2 <= 1
 0 <= assign_Eric_deal3 <= 1
 0 <= assign_Eric_deal4 <= 1
 0 <= assign_Eric_deal5 <= 1
 0 <= assign_Eric_deal6 <= 1
 0 <= assign_Eric_deal7 <= 1
 0 <= assign_Eric_deal8 <= 1
 0 <= assign_Eric_deal9 <= 1
 0 <= assign_Eric_deal10 <= 1
 0 <= x72 <= 1
 0 <= x74 <= 1
 0 <= x76 <= 1
 0 <= x78 <= 1
 0 <= x80 <= 1
 0 <= x82 <= 1
 0 <= x84 <= 1
 0 <= x86 <= 1
 0 <= x88 <= 1
 0 <= x90 <= 1
 0 <= x92 <= 1
 0 <= x94 <= 1
 0 <= x96 <= 1
 0 <= x98 <= 1
 0 <= x100 <= 1
 0 <= x102 <= 1
 0 <= x104 <= 1
 0 <= x106 <= 1
      x108 = 0
Binaries
 x1  assign_Mark_deal1  assign_Mark_deal2  assign_Mark_deal3 
 assign_Mark_deal4  assign_Mark_deal5  assign_Mark_deal6  assign_Mark_deal7 
 assign_Mark_deal8  assign_Mark_deal9  assign_Mark_deal10  x12 
 assign_Steve_deal1  assign_Steve_deal2  assign_Steve_deal3 
 assign_Steve_deal4  assign_Steve_deal5  assign_Steve_deal6 
 assign_Steve_deal7  assign_Steve_deal8  assign_Steve_deal9 
 assign_Steve_deal10  x24  assign_Paul_deal1  assign_Paul_deal2 
 assign_Paul_deal3  assign_Paul_deal4  assign_Paul_deal5  assign_Paul_deal6 
 assign_Paul_deal7  assign_Paul_deal8  assign_Paul_deal9  assign_Paul_deal10 
 x36  assign_John_deal1  assign_John_deal2  assign_John_deal3 
 assign_John_deal4  assign_John_deal5  assign_John_deal6  assign_John_deal7 
 assign_John_deal8  assign_John_deal9  assign_John_deal10  x48 
 assign_Jack_deal1  assign_Jack_deal2  assign_Jack_deal3  assign_Jack_deal4 
 assign_Jack_deal5  assign_Jack_deal6  assign_Jack_deal7  assign_Jack_deal8 
 assign_Jack_deal9  assign_Jack_deal10  x60  assign_Eric_deal1 
 assign_Eric_deal2  assign_Eric_deal3  assign_Eric_deal4  assign_Eric_deal5 
 assign_Eric_deal6  assign_Eric_deal7  assign_Eric_deal8  assign_Eric_deal9 
 assign_Eric_deal10  x72  x74  x76  x78  x80  x82  x84  x86  x88  x90  x92 
 x94  x96  x98  x100  x102  x104  x106 
End
'''
print( sample_string )
sample_string_bytes = sample_string.encode("ascii")
  
base64_bytes = base64.b64encode(sample_string_bytes)

base64_string = base64_bytes.decode("ascii")
print(f"Encoded string: {base64_string}")

In this sample, send an LP as inline data. Inline data is limited to small payloads and as an LP can be quite big, you must send the LP file using `OUTPUT_DATA_REFERENCES`.<BR>
`<connection_id>` is the connection id of the IBM Cloud Object Storage (infrastructure) created in your space.

In [ ]:
solve_payload = {
    "solve_parameters" : {
        "oaas.logTailEnabled":"true",
        "oaas.resultsFormat": "XML"
    },
    client.deployments.DecisionOptimizationMetaNames.INPUT_DATA: [
        {
            "id":"winloss.lp",
            "content" : base64_string
        }
    ],
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA_REFERENCES: [
        {
            "type" : "connection_asset",
            "id":".*\\.xml",
            "connection" : {
                "id" : "<connection_id>"
            },
            "location" : {
                "bucket" : "<bucket_name>",
                "file_name" : "${attachment_name}"
            }
        }
    ]
}
job_details = client.deployments.create_job(deployment_uid, solve_payload)
job_uid = client.deployments.get_job_id(job_details)

Display job status until it is completed.

The first job of a new deployment might take some time as a compute node must be started.

In [ ]:
from time import sleep

while job_details['entity']['decision_optimization']['status']['state'] not in ['completed', 'failed', 'canceled']:
    print(job_details['entity']['decision_optimization']['status']['state'] + '...')
    sleep(5)
    job_details=client.deployments.get_job_details(job_uid)

print(job_details['entity']['decision_optimization']['solve_state']['solve_status'])

You can launch a job with another LP file reusing the same deployment without waiting for compute node to be started. Here for simplicity the same LP file is reused.

In [ ]:
from time import sleep

job_details = client.deployments.create_job(deployment_uid, solve_payload)
job_uid = client.deployments.get_job_id(job_details)
while job_details['entity']['decision_optimization']['status']['state'] not in ['completed', 'failed', 'canceled']:
    print(job_details['entity']['decision_optimization']['status']['state'] + '...')
    sleep(5)
    job_details=client.deployments.get_job_details(job_uid)

print(job_details['entity']['decision_optimization']['solve_state']['solve_status'])

### Delete the deployment

You can delete deployment when no more LP jobs need to be executed.

In [ ]:
client.deployments.delete(deployment_uid)

<hr>
Copyright © 2019-2026. This notebook and its source code are released under the terms of the MIT License.